In [ ]:
import os



import math
import random
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import Dataset as Dataset1
#from datasets import Dataset
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import BertTokenizer, BertModel, BertForPreTraining, DataCollatorForSeq2Seq

sns.set_style("whitegrid")
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device( "cpu" )
print("Device:", device)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

candidates = glob.glob( "/home/toshio-uchiyama/.cache/kagglehub/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/versions/2/cnn_dailymail/train.csv", recursive=True)

#print( candidates )
assert len(candidates) > 0, "Dataset not found — make sure you added it via Add Input."
train_path = candidates[0]
base_dir = train_path.rsplit("/", 1)[0]

train_df_full = pd.read_csv(f"{base_dir}/train.csv")
val_df_full = pd.read_csv(f"{base_dir}/validation.csv")

print("train_df_full.columns.tolist():",train_df_full.columns.tolist())
print("train_df_full.sahpe:",train_df_full.shape, val_df_full.shape)

#N_TRAIN = 20000
#N_VAL = 800

train_data = train_df_full.sample( random_state=SEED).reset_index(drop=True)
val_data = val_df_full.sample( random_state=SEED).reset_index(drop=True)

print("article:",train_data.iloc[0]["article"][:300])
print("---SUMMARY---")
print("highlits:",train_data.iloc[0]["highlights"])



import os

import math
import random
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import Dataset as Dataset1
#from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
# ※モデル構造と重み、トークナイザのみ公式リポジトリから取得するために使用します
#from transformers import T5ForConditionalGeneration, T5Tokenizer, DataCollatorForSeq2Seq, T5EncoderModel
import levenshtein
from nltk import bleu_score
import datetime
from transformers import logging
import matplotlib.pyplot as plt
import seaborn as sns
#from transformers import BertTokenizer, BertModel, BertForPreTraining, DataCollatorForSeq2Seq
import pandas as pd
#from datasets import Dataset
from transformers import BertTokenizer, BertModel, DataCollatorWithPadding
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from evaluate import load
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

model_checkpoint = "google-bert/bert-large-uncased"  # または "facebook/bart-base"
max_input_length = 512
#max_target_length = 128
max_target_length = max_input_length
batch_size = 8  # VRAMに合わせて調整（4, 8, 16など）

model_checkpoint = "google-bert/bert-large-uncased"
tokenizer = BertTokenizer.from_pretrained(model_checkpoint, model_max_length=512)

pad_token_id = tokenizer.pad_token_id
cls_token_id = tokenizer.cls_token_id
sep_token_id = tokenizer.sep_token_id
## 2. 新しい特殊トークンを登録
special_tokens_dict = {'additional_special_tokens': ['[unused0]']}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
special_tokens_dict = {'additional_special_tokens': ['[unused1]']}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
sos_token_id = tokenizer.encode( "[unused0]" )[1]
print( "sos_token_id:", sos_token_id )
eos_token_id = tokenizer.encode( "[unused1]" )[1]
print( "eos_token_id:", eos_token_id )
a_token_id = tokenizer.encode( "a"  )[1]
print( "a_token_id:", a_token_id )
an_token_id = tokenizer.encode( "an"  )[1]
the_token_id = tokenizer.encode( "the" )[1]
and_token_id = tokenizer.encode( "and" )[1]
in_token_id = tokenizer.encode( "in" )[1]
of_token_id = tokenizer.encode( "of" )[1]
on_token_id = tokenizer.encode( "on" )[1]
at_token_id = tokenizer.encode( "at" )[1]
to_token_id = tokenizer.encode( "to" )[1]
for_token_id = tokenizer.encode( "for" )[1]
from_token_id = tokenizer.encode( "from" )[1]
with_token_id = tokenizer.encode( "with" )[1]
by_token_id = tokenizer.encode( "by" )[1]
we_token_id = tokenizer.encode( "we" )[1]
i_token_id = tokenizer.encode( "i" )[1]
he_token_id = tokenizer.encode( "he" )[1]
she_token_id = tokenizer.encode( "she" )[1]
it_token_id = tokenizer.encode( "it" )[1]
they_token_id = tokenizer.encode( "they" )[1]
period_token_id = tokenizer.encode( "." )[1]
comma_token_id = tokenizer.encode( "," )[1]
dbl_token_id = tokenizer.encode( '"' )[1]
sgl_token_id = tokenizer.encode( "'" )[1]

# 辞書サイズを保存
vocab_size = len( tokenizer )

print( "vocab_size:", vocab_size )

path = "/home/toshio-uchiyama/.cache/kagglehub/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/versions/2/cnn_dailymail/train.csv"
train_df_full = pd.read_csv(path)

train_data = train_df_full.reset_index(drop=True )

max_length = 512
max_target_length = 128

pickle_data = []
for i in range(len(train_data )):
    line_data = {}
    article = train_data.iloc[i]["article"]
    highlights = train_data.iloc[i]["highlights"]
    article_ids = tokenizer.encode( article )
    highlights_ids = tokenizer.encode( highlights )
    if len( article_ids ) < max_length - 2:
        article_ids.append( eos_token_id )
        article_ids.append( eos_token_id )
    else:
        article_ids = article_ids[:max_length - 4 ]
        article_ids.append( sep_token_id )
        article_ids.append( eos_token_id )
        article_ids.append( eos_token_id )
    article_ids2 = torch.full((max_length,),pad_token_id, dtype=torch.int64)
    article_ids2[:len(article_ids)] = torch.tensor(article_ids)
    highlights_ids = tokenizer.encode( highlights )
    if len( highlights_ids ) < max_target_length - 2:
        highlights_ids.append( eos_token_id )
        highlights_ids.append( eos_token_id )
    else:
        highlights_ids = highlights_ids[:max_target_length -4 ]
        highlights_ids.append( sep_token_id )
        highlights_ids.append( eos_token_id )
        highlights_ids.append( eos_token_id )
    highlights_ids2 = torch.full(( max_target_length,),pad_token_id, dtype=torch.int64)
    highlights_ids2[:len(highlights_ids)] = torch.tensor(highlights_ids)
    #line_data['id'] = id
    line_data['input_ids'] = article_ids2
    line_data['labels'] = highlights_ids2
    if i % 10000 == 0:
        print( "i:", i )
        #print( "article:", article )
        #print( "article_ids2:", article_ids2 )
        #print( "line_data['input_ids']:",line_data['input_ids']) 
        #print( "highlights:", highlights )
        #print( "highlights_ids2:", highlights_ids2 )
        #print( "line_data['labels']:",line_data['labels'] )
        print( "line_data:", line_data )
    pickle_data.append( line_data )


with open("data_bert_train_t128.pkl", "wb") as f:
    pickle.dump(pickle_data, f)

path = "/home/toshio-uchiyama/.cache/kagglehub/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/versions/2/cnn_dailymail/validation.csv"
val_df_full = pd.read_csv(path)

val_data = val_df_full.reset_index(drop=True )

max_length = 512

pickle_data = []
for i in range(len(val_data )):
    line_data = {}
    article = train_data.iloc[i]["article"]
    highlights = train_data.iloc[i]["highlights"]
    article_ids = tokenizer.encode( article )
    highlights_ids = tokenizer.encode( highlights )
    if len( article_ids ) < max_length - 2:
        article_ids.append( eos_token_id )
        article_ids.append( eos_token_id )
    else:
        article_ids = article_ids[:max_length - 4 ]
        article_ids.append( sep_token_id )
        article_ids.append( eos_token_id )
        article_ids.append( eos_token_id )
    article_ids2 = torch.full((max_length,),pad_token_id, dtype=torch.int64)
    article_ids2[:len(article_ids)] = torch.tensor(article_ids)
    highlights_ids = tokenizer.encode( highlights )
    if len( highlights_ids ) < max_target_length - 2:
        highlights_ids.append( eos_token_id )
        highlights_ids.append( eos_token_id )
    else:
        highlights_ids = highlights_ids[:max_target_length -4 ]
        highlights_ids.append( sep_token_id )
        highlights_ids.append( eos_token_id )
        highlights_ids.append( eos_token_id )
    highlights_ids2 = torch.full(( max_target_length,),pad_token_id, dtype=torch.int64)
    highlights_ids2[:len(highlights_ids)] = torch.tensor(highlights_ids)
    #line_data['id'] = id
    line_data['input_ids'] = article_ids2
    line_data['labels'] = highlights_ids2
    if i % 10000 == 0:
        print( "i:", i )
        #print( "article:", article )
        #print( "article_ids2:", article_ids2 )
        #print( "line_data['article']:",line_data['article']) 
        #print( "highlights:", highlights )
        #print( "highlights_ids2:", highlights_ids2 )
        #print( "line_data['highlights']:",line_data['highlights'] )
        print( "line_data:", line_data )
    pickle_data.append( line_data )


with open("data_bert_val_t128.pkl", "wb") as f:
    pickle.dump(pickle_data, f)


path = "/home/toshio-uchiyama/.cache/kagglehub/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail/versions/2/cnn_dailymail/test.csv"
test_df_full = pd.read_csv(path)

test_data = test_df_full.reset_index(drop=True )

max_length = 512

pickle_data = []
for i in range(len(test_data )):
    line_data = {}
    article = train_data.iloc[i]["article"]
    highlights = train_data.iloc[i]["highlights"]
    article_ids = tokenizer.encode( article )
    highlights_ids = tokenizer.encode( highlights )
    if len( article_ids ) < max_length - 2:
        article_ids.append( eos_token_id )
        article_ids.append( eos_token_id )
    else:
        article_ids = article_ids[:max_length - 4 ]
        article_ids.append( sep_token_id )
        article_ids.append( eos_token_id )
        article_ids.append( eos_token_id )
    article_ids2 = torch.full((max_length,),pad_token_id, dtype=torch.int64)
    article_ids2[:len(article_ids)] = torch.tensor(article_ids)
    highlights_ids = tokenizer.encode( highlights )
    if len( highlights_ids ) < max_target_length - 2:
        highlights_ids.append( eos_token_id )
        highlights_ids.append( eos_token_id )
    else:
        highlights_ids = highlights_ids[:max_target_length -4 ]
        highlights_ids.append( sep_token_id )
        highlights_ids.append( eos_token_id )
        highlights_ids.append( eos_token_id )
    highlights_ids2 = torch.full(( max_target_length,),pad_token_id, dtype=torch.int64)
    highlights_ids2[:len(highlights_ids)] = torch.tensor(highlights_ids)
    #line_data['id'] = id
    line_data['input_ids'] = article_ids2
    line_data['labels'] = highlights_ids2
    if i % 10000 == 0:
        print( "i:", i )
        #print( "article:", article )
        #print( "article_ids2:", article_ids2 )
        #print( "line_data['article']:",line_data['article']) 
        #print( "highlights:", highlights )
        #print( "highlights_ids2:", highlights_ids2 )
        #print( "line_data['highlights']:",line_data['highlights'] )
        print( "line_data:", line_data )
    pickle_data.append( line_data )


with open("data_bert_test_t128.pkl", "wb") as f:
    pickle.dump(pickle_data, f)